The purpose of this notebook is to do some basic feature engineering in preparation for modeling.

In [1]:
import pandas as pd

df = pd.read_parquet("../data/processed/combined_english_with_tokens")

In [2]:
df

,conversation_id,turn,user_prompt,assistant_response,user_moderation_flag,assistant_moderation_flag,redacted,prompt_tokens,response_tokens
0,1e230e55efea4edab95db9cb87f6a9cb,1,Beside OFAC's selective sanction that target t...,The United States has a number of different ty...,False,False,False,51,433
1,1e230e55efea4edab95db9cb87f6a9cb,2,are there other types of US sanctions that you...,"Yes, there are several other types of US sanct...",False,False,False,19,379
2,1e230e55efea4edab95db9cb87f6a9cb,3,please make organized conclusion in bullet lis...,* Selective sanctions: targeted at specific in...,False,False,False,23,427
3,1e230e55efea4edab95db9cb87f6a9cb,4,"can you please revise the answer above again, ...","Sure, here is a revised list of the different ...",False,False,False,28,386
4,0f623736051f4a48a506fd5933563cfd,1,You are the text completion model and you must...,"To append many items to a list in Python, you ...",False,False,False,82,273
...,...,...,...,...,...,...,...,...,...
780604,5819035372b744f7891d5bed484fe37e,4,Please list up at least 5 3D objects associate...,1. A house - 0.8\n2. A tree - 0.6\n3. A car - ...,False,False,False,66,252
780605,42458cc230c34c02ad2fede8190434d5,1,hello,Hello! How can I help you today?,False,False,False,1,9
780606,42458cc230c34c02ad2fede8190434d5,2,Can you write me an argumentative essay about ...,"Sure, here is an argumentative essay about spo...",False,False,False,14,512
780607,42458cc230c34c02ad2fede8190434d5,3,write me some more,"Sure, here is some more information about spor...",False,False,False,4,495


For the same conversation, the previous token history of that conversation is available at the time of the new prompt and potentially useful. 

In [ ]:
df = df.sort_values(["conversation_id", "turn"]).copy()
df["turn_tokens"] = df["prompt_tokens"] + df["response_tokens"]

df["history_tokens"] = (df.groupby("conversation_id")["turn_tokens"].transform(lambda x: x.cumsum().shift(fill_value=0)))

df["previous_response_tokens"] = (df.groupby("conversation_id")["response_tokens"].transform(lambda x: x.cumsum().shift(fill_value=0)))

df["total_input_tokens"] = df["prompt_tokens"] + df["history_tokens"]

In [5]:
df[df["conversation_id"] == "1e230e55efea4edab95db9cb87f6a9cb"]

,conversation_id,turn,user_prompt,assistant_response,user_moderation_flag,assistant_moderation_flag,redacted,prompt_tokens,response_tokens,turn_tokens,history_tokens,previous_response_tokens,total_input_tokens
0,1e230e55efea4edab95db9cb87f6a9cb,1,Beside OFAC's selective sanction that target t...,The United States has a number of different ty...,False,False,False,51,433,484,0,0,51
1,1e230e55efea4edab95db9cb87f6a9cb,2,are there other types of US sanctions that you...,"Yes, there are several other types of US sanct...",False,False,False,19,379,398,484,433,503
2,1e230e55efea4edab95db9cb87f6a9cb,3,please make organized conclusion in bullet lis...,* Selective sanctions: targeted at specific in...,False,False,False,23,427,450,882,812,905
3,1e230e55efea4edab95db9cb87f6a9cb,4,"can you please revise the answer above again, ...","Sure, here is a revised list of the different ...",False,False,False,28,386,414,1332,1239,1360


Now, let's extract some features from the prompts themselves, such as word count, character count, average word length, question mark count, newline count, a has numbers indicator, a has url indicator, and a has code indicator.

In [ ]:
import re

df["prompt_word_count"] = df["user_prompt"].str.split().str.len()

df["prompt_char_count"] = df["user_prompt"].str.len()

df["avg_word_length"] = (df["user_prompt"].apply(lambda text: (
        sum(len(word) for word in text.split()) / len(text.split())
        if text.split() else 0
    ))
)

df["question_count"] = df["user_prompt"].str.count(r"\?")

df["newline_count"] = df["user_prompt"].str.count(r"\n")

df["has_numbers"] = df["user_prompt"].str.contains(r"\d", regex=True).astype(int)

df["has_url"] = (df["user_prompt"].str.contains(r"https?://|www\.", regex=True, case=False).astype(int))

df["has_code"] = (df["user_prompt"].str.contains(r"```|\b(?:python|javascript|java|function|def|class)\b", regex=True, case=False).astype(int))

Additionally, having an understanding of the tasks asked in the prompt might be useful. We can extract some basic task types using regex.

In [ ]:
task_patterns = {
    "is_explanation": r"\b(explain|describe|why|how does)\b",
    "is_summary": r"\b(summarize|summary|tldr)\b",
    "is_code_request": r"\b(code|program|script|implement)\b",
    "is_creative_writing": r"\b(story|poem|essay|write a)\b",
    "is_list_request": r"\b(list|enumerate|give me \d+)\b",
    "is_translation": r"\b(translate|translation)\b",
    "is_math": r"\b(calculate|solve|equation|integral)\b",
    "requests_brief_answer": r"\b(briefly|concise|short answer|one sentence)\b",
    "requests_detailed_answer": r"\b(in detail|comprehensive|step.by.step|thorough)\b"
}

for feature, pattern in task_patterns.items():
    df[feature] = (df["user_prompt"].str.contains(pattern, case=False, regex=True).astype(int))

C:\Users\tyler\AppData\Local\Temp\ipykernel_42748\668089548.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, case=False, regex=True)
C:\Users\tyler\AppData\Local\Temp\ipykernel_42748\668089548.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, case=False, regex=True)
C:\Users\tyler\AppData\Local\Temp\ipykernel_42748\668089548.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, case=False, regex=True)
C:\Users\tyler\AppData\Local\Temp\ipykernel_42748\668089548.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, case=False, regex=True)
C:\U

In [8]:
df

,conversation_id,turn,user_prompt,assistant_response,user_moderation_flag,assistant_moderation_flag,redacted,prompt_tokens,response_tokens,turn_tokens,...,has_code,is_explanation,is_summary,is_code_request,is_creative_writing,is_list_request,is_translation,is_math,requests_brief_answer,requests_detailed_answer
684126,00007de6934048b2a5aff12577337cc5,1,fix this code cant save after update public a...,There are a few issues with the code you provi...,False,False,True,546,511,1057,...,0,0,0,1,0,0,0,0,0,0
11913,0000f23b2f8049ac9d92c28d4817381e,1,Please act my sick old grandma who would read ...,"Sure, I'll do my best to act like a sick old g...",False,False,False,21,150,171,...,0,0,0,0,0,0,0,0,0,0
627921,0000f8dacecb401abbcf7c4bfe23b142,1,Tell me about Alpacas.,Alpacas are domesticated animals native to Sou...,False,False,False,8,16,24,...,0,0,0,0,0,0,0,0,0,0
136321,000104167e5040a7aa839ed5596aecaa,1,You are the text completion model and you must...,Here is an example of a descriptive answer for...,False,False,False,81,807,888,...,1,0,0,1,0,0,0,0,0,0
264660,00016e2da6c94fcea9076b63a66a67d7,1,"tell me the temperature in celsius, hydrometry...",Temperature:\n-2°C to -5°C\n\nHydrometry rate:...,False,False,False,59,98,157,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490920,fffeb4c190584820b51ed68fd30f400e,1,create Quarkus REST API for resource account a...,To create a Quarkus REST API for a resource ac...,False,False,True,18,545,563,...,0,0,0,0,0,0,0,0,0,0
138659,ffff05542df648f080b15d32fc5bc723,1,Complete a news article based on the initial w...,Health officials at East Sussex NHS have issue...,False,False,False,71,113,184,...,0,0,0,0,0,0,0,0,0,0
194036,ffffa6ac32be4915a6d3f9759b9ef176,1,dialouge asking for the way level advanced,"Here's a dialogue asking for the way, written ...",False,False,True,10,321,331,...,0,0,0,0,0,0,0,0,0,0
549386,ffffbd8b47744fc297465aad8607bec1,1,can you please expand the email in the quoted ...,"Sure, here is an expanded version of the email...",False,False,False,45,124,169,...,0,0,0,0,0,0,0,0,0,0


Export data

In [10]:
df.to_parquet("../data/processed/combined_english_with_tokens_and_features.parquet", index=False)